In [1]:
import pandas as pd

orders = pd.read_csv('../raw/orders.csv')
orders.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0


In [2]:
orders.shape

(3421083, 7)

In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), str(1)
memory usage: 182.7 MB


In [5]:
orders[orders['days_since_prior_order'].isnull()]['order_number'].unique()

array([1])

In [6]:
orders[orders['days_since_prior_order'].isnull()]['order_number'].unique()

array([1])

In [7]:
products = pd.read_csv('../raw/products.csv')
aisles = pd.read_csv('../raw/aisles.csv')
departments = pd.read_csv('../raw/departments.csv')
order_products_prior = pd.read_csv('../raw/order_products__prior.csv')
order_products_train = pd.read_csv('../raw/order_products__train.csv')

In [8]:
print("products:", products.shape)
print("aisles:", aisles.shape)
print("departments:", departments.shape)
print("order_products_prior:", order_products_prior.shape)
print("order_products_train:", order_products_train.shape)

products: (49688, 4)
aisles: (134, 2)
departments: (21, 2)
order_products_prior: (32434489, 4)
order_products_train: (1384617, 4)


In [9]:
order_products_train.head()

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1


In [10]:
order_products_train_named = order_products_train.merge(products, on='product_id', how='left')
order_products_train_named.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id
0,1,49302,1,1,Bulgarian Yogurt,120,16
1,1,11109,2,1,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16
2,1,10246,3,0,Organic Celery Hearts,83,4
3,1,49683,4,0,Cucumber Kirby,83,4
4,1,43633,5,1,Lightly Smoked Sardines in Olive Oil,95,15


In [11]:
order_products_full = order_products_train_named.merge(aisles, on='aisle_id', how='left')
order_products_full = order_products_full.merge(departments, on='department_id', how='left')
order_products_full.head()

,order_id,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,aisle,department
0,1,49302,1,1,Bulgarian Yogurt,120,16,yogurt,dairy eggs
1,1,11109,2,1,Organic 4% Milk Fat Whole Milk Cottage Cheese,108,16,other creams cheeses,dairy eggs
2,1,10246,3,0,Organic Celery Hearts,83,4,fresh vegetables,produce
3,1,49683,4,0,Cucumber Kirby,83,4,fresh vegetables,produce
4,1,43633,5,1,Lightly Smoked Sardines in Olive Oil,95,15,canned meat seafood,canned goods


In [12]:
order_products_full.to_csv('../raw/order_products_full.csv', index=False)

In [13]:
order_products_full['department'].value_counts()

department
produce            409087
dairy eggs         217051
snacks             118862
beverages          114046
frozen             100426
pantry              81242
bakery              48394
canned goods        46799
deli                44291
dry goods pasta     38713
household           35986
meat seafood        30307
breakfast           29500
personal care       21570
babies              14941
international       11902
missing              8251
alcohol              5598
pets                 4497
other                1795
bulk                 1359
Name: count, dtype: int64

In [14]:
order_products_full.groupby('department')['reordered'].mean().sort_values(ascending=False)

department
dairy eggs         0.674966
produce            0.664617
beverages          0.658155
bakery             0.634211
pets               0.630198
deli               0.617891
alcohol            0.606824
meat seafood       0.590854
snacks             0.581363
bulk               0.578366
breakfast          0.571661
frozen             0.559297
babies             0.541062
dry goods pasta    0.487821
canned goods       0.486805
household          0.427166
other              0.388301
missing            0.381530
international      0.379936
pantry             0.363088
personal care      0.337089
Name: reordered, dtype: float64

In [15]:
from scipy.stats import chi2_contingency

# Sirf dono departments ka data filter karo
subset = order_products_full[order_products_full['department'].isin(['dairy eggs', 'personal care'])]

# Contingency table banao (department vs reordered ka cross-tab)
contingency_table = pd.crosstab(subset['department'], subset['reordered'])
print(contingency_table)

reordered          0       1
department                  
dairy eggs     70549  146502
personal care  14299    7271


In [16]:
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 9773.525902313515
p-value: 0.0
Degrees of freedom: 1


In [17]:
from scipy.stats import ttest_ind

produce_cart_order = order_products_full[order_products_full['department'] == 'produce']['add_to_cart_order']
snacks_cart_order = order_products_full[order_products_full['department'] == 'snacks']['add_to_cart_order']

print("Produce average position:", produce_cart_order.mean())
print("Snacks average position:", snacks_cart_order.mean())

t_stat, p_value = ttest_ind(produce_cart_order, snacks_cart_order)
print("T-statistic:", t_stat)
print("p-value:", p_value)

Produce average position: 8.431047674455556
Snacks average position: 9.562097221988525
T-statistic: -47.88018975733
p-value: 0.0


In [18]:
from scipy.stats import f_oneway

departments_to_compare = ['produce', 'dairy eggs', 'snacks', 'beverages', 'frozen']

groups = [
    order_products_full[order_products_full['department'] == dept]['add_to_cart_order']
    for dept in departments_to_compare
]

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("p-value:", p_value)

# Har department ka average bhi dekh lete hain reference ke liye
for dept in departments_to_compare:
    avg = order_products_full[order_products_full['department'] == dept]['add_to_cart_order'].mean()
    print(f"{dept}: {avg:.2f}")

F-statistic: 2494.2872915953244
p-value: 0.0
produce: 8.43
dairy eggs: 7.88
snacks: 9.56
beverages: 7.14
frozen: 9.44


## Statistical Insights — Phase 4

### 1. Department vs Reorder Behavior (Chi-Square Test)
**Question:** Kya reorder rate department ke basis pe significantly differ karta hai?
- Dairy Eggs: 67.5% reorder rate | Personal Care: 33.7% reorder rate
- Chi-square = 9773.53, p < 0.001 → **Statistically significant**
- **Insight:** Fresh/perishable categories (dairy, produce) habit-driven hain; personal care items infrequent/planned purchases hain.

### 2. Cart Position — Produce vs Snacks (T-Test)
**Question:** Kya produce items snacks se pehle cart mein add hote hain?
- Produce avg position: 8.43 | Snacks avg position: 9.56
- T-statistic = -47.88, p < 0.001 → **Statistically significant**
- **Insight:** Essential/planned items pehle add hote hain, discretionary items baad mein.

### 3. Cart Position Across 5 Departments (ANOVA)
**Question:** Kya cart position multiple departments mein differ karta hai?
- Beverages: 7.14 | Dairy Eggs: 7.88 | Produce: 8.43 | Frozen: 9.44 | Snacks: 9.56
- F-statistic = 2494.29, p < 0.001 → **Statistically significant**
- **Insight:** Beverages/Dairy = planned purchases (early cart); Snacks/Frozen = impulse/secondary purchases (late cart).

### Business Takeaway
Data confirms two behavioral patterns: (1) purchase frequency/loyalty varies strongly by category type — fresh/perishable > discretionary; (2) shopping planning sequence follows a "essentials-first" pattern. These insights could inform personalization features — e.g., reminder nudges for high-reorder categories, or cart-suggestion ordering aligned with natural shopping sequence.

In [19]:
import sqlite3

# Database connection banao (agar file exist nahi karti, naye banegi)
conn = sqlite3.connect('../insights_agent.db')

# Cleaned/merged table ko database mein daalo
order_products_full.to_sql('order_products_full', conn, if_exists='replace', index=False)

# Baaki core tables bhi daal do
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
aisles.to_sql('aisles', conn, if_exists='replace', index=False)
departments.to_sql('departments', conn, if_exists='replace', index=False)

print("Database created successfully!")

Database created successfully!


In [20]:
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('order_products_full',), ('orders',), ('products',), ('aisles',), ('departments',)]


In [21]:
query = """
SELECT department, COUNT(*) as total_orders, AVG(reordered) as reorder_rate
FROM order_products_full
GROUP BY department
ORDER BY reorder_rate DESC
LIMIT 5
"""

result = pd.read_sql_query(query, conn)
print(result)

   department  total_orders  reorder_rate
0  dairy eggs        217051      0.674966
1     produce        409087      0.664617
2   beverages        114046      0.658155
3      bakery         48394      0.634211
4        pets          4497      0.630198
